In [16]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm, trange
from torch.utils.data import DataLoader
import os
import json

In [17]:
def tokens_to_ids(tokens, vocab):
    return [
        vocab.get(token.lower(), vocab["<unk>"])
        for token in tokens
    ]

In [18]:
class GeniaDataset(torch.utils.data.Dataset):
    def __init__(self, data, vocab, label2id, max_span_len=6):
        self.data = data
        self.vocab = vocab
        self.label2id = label2id
        self.max_span_len = max_span_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item["tokens"]
        entities = item["entities"]

        token_ids = tokens_to_ids(tokens, self.vocab)
        spans = generate_spans(len(tokens), self.max_span_len)
        labels = create_span_labels(entities, spans, self.label2id)
        max_spans = 200
        if len(spans) > max_spans:
            spans = spans[:max_spans]
            labels = labels[:max_spans]
        return {
            "token_ids": torch.tensor(token_ids),
            "spans": torch.tensor(spans),
            "labels": torch.tensor(labels)
        }

In [19]:
label2id = {
    "NONE": 0,
    "DNA": 1,
    "RNA": 2,
    "cell_type": 3,
    "protein": 4,
    "cell_line": 5
}

id2label = {v: k for k, v in label2id.items()}
def generate_spans(n, max_span_len=10):
    spans = []
    for i in range(n):
        for j in range(i+1, min(n+1, i+max_span_len+1)):
            spans.append((i, j))
    return spans


def create_span_labels(entities, spans, label2id):
    gt = {
        (ent["start"], ent["end"]): ent["type"]
        for ent in entities
    }

    labels = []
    for span in spans:
        if span in gt:
            labels.append(label2id[gt[span]])
        else:
            labels.append(label2id["NONE"])
    
    return labels

    

In [20]:
class EmbeddingLayer(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        num_embeddings, emb_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(num_embeddings, emb_dim)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = True  # you can freeze later

    def forward(self, token_ids):
        return self.embedding(token_ids)

In [21]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        outputs, _ = self.lstm(x)
        return outputs  # (B, T, H)

In [22]:
def get_span_representations(H_batch, spans_batch):
    """
    H_batch: (B, T, H)
    spans_batch: (B, max_spans, 2)
    """
    B, T, H = H_batch.shape
    max_spans = spans_batch.shape[1]
    
    # Extract start and end indices
    start_indices = spans_batch[:, :, 0] # (B, max_spans)
    end_indices = spans_batch[:, :, 1]   # (B, max_spans)
    
    # Create a mask for valid spans (i=0, j=0 is used for padding)
    valid_mask = (end_indices > 0) # (B, max_spans)
    
    # Batch indices for advanced indexing
    batch_indices = torch.arange(B, device=H_batch.device).view(B, 1).expand(B, max_spans)
    
    # Start and End representations
    safe_start = start_indices.clamp(0, T - 1)
    safe_end = (end_indices - 1).clamp(0, T - 1)
    
    start_reprs = H_batch[batch_indices, safe_start]
    end_reprs = H_batch[batch_indices, safe_end]
    
    # Mean representations using cumsum
    H_cumsum = torch.cumsum(H_batch, dim=1)
    H_cumsum = torch.cat([torch.zeros(B, 1, H, device=H_batch.device), H_cumsum], dim=1)
    
    span_sums = H_cumsum[batch_indices, end_indices] - H_cumsum[batch_indices, start_indices]
    span_lens = (end_indices - start_indices).float().clamp(min=1.0).unsqueeze(-1)
    mean_reprs = span_sums / span_lens
    
    # Concatenate
    span_reprs = torch.cat([start_reprs, end_reprs, mean_reprs], dim=-1)
    
    # Mask invalid spans
    span_reprs = span_reprs * valid_mask.unsqueeze(-1)
    
    return span_reprs

In [23]:
class SpanClassifier(nn.Module):
    def __init__(self, input_dim, num_labels):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(input_dim, num_labels)
        )

    def forward(self, span_repr):
        return self.mlp(span_repr)

In [24]:
class SpanNERModel(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_labels):
        super().__init__()
        emb_dim = embedding_matrix.shape[1]

        self.embedding = EmbeddingLayer(embedding_matrix)
        self.encoder = Encoder(emb_dim, hidden_dim)

        self.classifier = SpanClassifier(
            input_dim=hidden_dim * 3,
            num_labels=num_labels
        )

    def forward(self, token_ids, spans):
        """
        token_ids: (B, T)
        spans: (B, max_spans, 2)
        """
        # token_ids already has the batch dimension (B, T), so no unsqueeze needed!
        x = self.embedding(token_ids)   # (B, T, E)
        H = self.encoder(x)             # (B, T, H)

        span_repr = get_span_representations(H, spans) # (B, max_spans, 3H)
        logits = self.classifier(span_repr)            # (B, max_spans, num_labels)

        return logits

In [25]:
def compute_f1(all_preds, all_labels):
    tp = 0
    fp = 0
    fn = 0
    
    # Calculate micro-averaged F1, ignoring the "NONE" class (0)
    for pred, true in zip(all_preds, all_labels):
        if pred != 0 and pred == true:
            tp += 1
        elif pred != 0 and pred != true:
            fp += 1
        elif true != 0 and pred != true:
            fn += 1

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    token_ids = [item["token_ids"] for item in batch]
    spans = [item["spans"] for item in batch]
    labels = [item["labels"] for item in batch]

    # pad_sequence turns the list of 1D tensors into one cohesive 2D Tensor
    padded_token_ids = pad_sequence(token_ids, batch_first=True, padding_value=0)
    padded_spans = pad_sequence(spans, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "token_ids": padded_token_ids,
        "spans": padded_spans,
        "labels": padded_labels
    }

In [27]:
# 1. Load Data
with open("genia/genia_train_dev.json") as f:
    data = json.load(f)
with open("genia/vocab.json") as f:
    vocab = json.load(f)
# 2. Split Data
train_data, dev_data = train_test_split(data, test_size=0.1, random_state=42)

# 3. Load Embeddings and Vocab (Assuming you have a way to load vocab)
embedding_matrix = np.load("genia/vocab_embed_bio.npy")
# vocab = ... (YOU NEED TO DEFINE/LOAD THIS HERE)

# 4. Create Datasets and Dataloaders
train_dataset = GeniaDataset(train_data, vocab, label2id)
dev_dataset = GeniaDataset(dev_data, vocab, label2id)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SpanNERModel(
    embedding_matrix=embedding_matrix,
    hidden_dim=256,
    num_labels=len(label2id)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# Initialize loss function to specifically ignore the -100 padding
criterion = nn.CrossEntropyLoss(ignore_index=-100)

def train_epoch(model, dataloader):
    model.train()
    total_loss = 0

    loop = tqdm(dataloader, desc="Training", leave=False)

    for batch in loop:
        optimizer.zero_grad()
        
        # 1. Move to device directly from the batched dictionary
        token_ids = batch["token_ids"].to(device)
        spans = batch["spans"].to(device)
        labels = batch["labels"].to(device)

        # 2. Forward pass (Logits shape: [B, max_spans, num_labels])
        logits = model(token_ids, spans) 

        # 3. Flatten logits and labels for CrossEntropyLoss
        # Logits become: (B * max_spans, num_labels)
        # Labels become: (B * max_spans)
        logits_flat = logits.view(-1, logits.size(-1))
        labels_flat = labels.view(-1)

        # 4. Calculate Loss
        loss = criterion(logits_flat, labels_flat)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return total_loss / len(dataloader)
def get_predictions(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            # Send the whole batch to the GPU at once
            token_ids = batch["token_ids"].to(device)
            spans = batch["spans"].to(device)
            labels = batch["labels"].to(device)

            # Get batched predictions
            logits = model(token_ids, spans)            # (B, max_spans, num_labels)
            preds = torch.argmax(logits, dim=-1)        # (B, max_spans)

            # Flatten them just like we did in the training loop
            preds_flat = preds.view(-1)
            labels_flat = labels.view(-1)

            # CRITICAL: Create a mask to ignore the -100 padded spans
            valid_mask = (labels_flat != -100)
            
            # Apply the mask
            valid_preds = preds_flat[valid_mask]
            valid_labels = labels_flat[valid_mask]

            # Store the valid predictions
            all_preds.extend(valid_preds.cpu().tolist())
            all_labels.extend(valid_labels.cpu().tolist())

    return all_preds, all_labels

In [29]:
EPOCHS = 5

for epoch in trange(EPOCHS, desc="Epochs"):
    train_loss = train_epoch(model, train_loader)

    preds, labels = get_predictions(model, dev_loader)
    p, r, f1 = compute_f1(preds, labels)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}")
    print("------")

Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Training:   0%|          | 0/3756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/418 [00:00<?, ?it/s]


Epoch 1
Train Loss: 0.0382
Precision: 0.8067, Recall: 0.7252, F1: 0.7638
------


Training:   0%|          | 0/3756 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/418 [00:00<?, ?it/s]


Epoch 4
Train Loss: 0.0158
Precision: 0.7833, Recall: 0.7926, F1: 0.7880
------


Training:   0%|          | 0/3756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/418 [00:00<?, ?it/s]


Epoch 5
Train Loss: 0.0128
Precision: 0.7716, Recall: 0.8091, F1: 0.7899
------
